In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
import pandas as pd
from sklearn.metrics import roc_auc_score

DRIVE_DIR = "/content/drive/MyDrive/산공종설"
OOF = f"{DRIVE_DIR}/results/resnet18_fixedcrop512_all4000_fast_seeds0/oof_predictions.csv"
assert os.path.exists(OOF), "고정 크롭 결과 파일이 없음: Drive 경로를 확인"

# 우리 실험에 쓴 3,425장 목록(분할표)만 zip에서 꺼내기
!unzip -o -j -q "{DRIVE_DIR}/keep3425_masked_500.zip" folds.csv -d /content/keep_list
keep = set(pd.read_csv("/content/keep_list/folds.csv", encoding="utf-8-sig")["source"])
print("선별 목록:", len(keep), "장")   # 3425가 나와야 함

# 고정 크롭 실험(③)이 저장해 둔 사진별 예측
oof = pd.read_csv(OOF, encoding="utf-8-sig")

def score(d):
    y, p = d["label_id"], d["pred"]
    return {"사진수": len(d), "정확도": round((y == p).mean(), 3),
            "언더필재현율": round(p[y == 1].mean(), 3), "정상재현율": round(1 - p[y == 0].mean(), 3),
            "AUROC": round(roc_auc_score(y, d["prob_underfill"]), 3)}

rows = []
for mode, d in oof.groupby("mode"):
    for name, sub in (("전체 4000장", d),
                      ("선별 3425장만", d[d["source"].isin(keep)]),
                      ("제외된 575장만", d[~d["source"].isin(keep)])):
        rows.append({"모드": mode, "대상": name, **score(sub)})
print(pd.DataFrame(rows).to_string(index=False))
print("\n비교용 - 우리 방식(이동·마스킹, 선별 3425장) finetune: 정확도 0.976 / 언더필 0.965 / 정상 0.987 / AUROC 0.992")